# Phần 2 – Khai phá quy trình trên BPMN và nhật ký của text2BPMN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Punpunpuong22/final_report_texttobpmn_proces_mining/blob/main/notebooks/Part2_ProcessMining.ipynb)

Notebook chạy pipeline tám bước trên một tệp BPMN do text2BPMN xuất:

| Bước | Câu hỏi | Đầu ra |
|---|---|---|
| 1 | BPMN có hợp lệ về hành vi (sound) không? | Bảng phần tử, soundness, lỗi cấu trúc |
| 2 | Lấy nhật ký sự kiện | Log CSV/XES theo kịch bản, kiểm tra chất lượng |
| 3 | Quy trình chạy thế nào? | KPI, tỷ trọng thời gian làm việc, biến thể |
| 4 | Khám phá mô hình từ log | DFG, Inductive, Heuristics, bốn chỉ số chất lượng |
| 5 | Vận hành có đúng BPMN và quy định? | Alignment, vi phạm luật |
| 6 | Điểm nghẽn ở đâu? | Chờ và xử lý, tải từng làn, hàng đợi |
| 7 | Yếu tố nào gắn với trễ hạn? | Cây quyết định |
| 8 | TO-BE có tốt hơn không, nhờ thay đổi nào? | BPMN TO-BE, so sánh có kiểm định, ablation |

**Chạy trên Colab:** Runtime → Run all. **Chạy trên Kaggle:** bật *Internet* trong Settings của notebook rồi Run all.

> Nhật ký mặc định là **dữ liệu mô phỏng** với tham số giả định. Kết quả minh hoạ phương pháp, không phải số liệu vận hành thật.

## 0. Cài đặt môi trường

In [ ]:
# Kho GitHub của nhóm. Ô này tự clone kho nếu chưa có mã nguồn.
REPO_URL = "https://github.com/Punpunpuong22/final_report_texttobpmn_proces_mining.git"
REPO_NAME = "final_report_texttobpmn_proces_mining"

import glob, os, shutil, subprocess, sys, zipfile
os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "False"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/working")

def find_repo():
    """Tìm thư mục chứa mã nguồn: thư mục hiện tại, thư mục cha, bản đã clone, tệp zip tải lên, hoặc dataset Kaggle."""
    for d in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/" + REPO_NAME, "/kaggle/working/" + REPO_NAME]:
        if os.path.isdir(os.path.join(d, "src", "text2bpmn_mining")):
            return d
    base = "/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()
    for z in glob.glob(os.path.join(base, "*.zip")) + glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            f.extractall(base)
    for d in glob.glob(os.path.join(base, "**", "src", "text2bpmn_mining"), recursive=True) + \
             glob.glob("/kaggle/input/**/src/text2bpmn_mining", recursive=True):
        root = os.path.dirname(os.path.dirname(d))
        if root.startswith("/kaggle/input"):                 # dataset Kaggle chỉ đọc → chép sang /kaggle/working
            dst = "/kaggle/working/" + REPO_NAME
            shutil.copytree(root, dst, dirs_exist_ok=True)
            root = dst
        return root
    if REPO_URL:
        dst = os.path.join(base, REPO_NAME)
        subprocess.run(["git", "clone", "-q", REPO_URL, dst], check=True)
        return dst
    raise SystemExit("Không tìm thấy mã nguồn. Sửa REPO_URL, hoặc tải tệp zip của kho lên (Colab: thanh Files; "
                     "Kaggle: Add Data) rồi chạy lại ô này.")

ROOT = find_repo()
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

try:
    import pm4py  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if shutil.which("dot") is None and (IN_COLAB or IN_KAGGLE):   # graphviz để vẽ DFG và BPMN
    subprocess.run("apt-get -qq update && apt-get -qq install -y graphviz > /dev/null", shell=True)

import pandas as pd
import pm4py
import text2bpmn_mining as t2b
try:
    from IPython.display import Image, display
except ImportError:                                              # chạy ngoài notebook
    display, Image = print, (lambda filename: filename)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 170)

def show(path):
    if os.path.exists(path):
        display(Image(filename=path))

print("Thư mục:", ROOT, "| pm4py", pm4py.__version__, "| text2bpmn_mining", t2b.__version__)

## Cấu hình

Chọn tệp BPMN cần phân tích. Ba tệp mẫu do text2BPMN xuất nằm trong `data/bpmn/`.
Nếu đã có nhật ký do text2BPMN xuất, đặt các tệp CSV/XES vào `data/part1_logs/<tên tệp BPMN>/`
(mỗi tệp là một kịch bản); pipeline sẽ dùng nhật ký đó thay cho bộ mô phỏng.

Mọi tham số trong `Config` là giả định và có thể sửa: số hồ sơ, thời gian từng bước, số người mỗi làn,
kịch bản, thời hạn, luật nghiệp vụ, bước đề xuất tự động hoá.

In [ ]:
BPMN_FILE = "data/bpmn/quy-trinh-nghi-phep.bpmn"     # hoặc quy-trinh-tuyen-dung.bpmn, quy-trinh-dat-hang.bpmn

cfg = t2b.Config(
    seed=42,
    n_cases=200,            # số hồ sơ mỗi kịch bản
    n_rep=10,               # số lần lặp khi so sánh AS-IS và TO-BE (giảm xuống 3 để chạy nhanh)
    sla_minutes=None,       # thời hạn (phút); None = 1,5 × P90 thời gian lưu trú của kịch bản đầu
    task_minutes={},        # ghi đè thời gian từng bước, vd {"Xem xét đơn": 25}
    capacity=None,          # số người mỗi làn, vd {"Quản lý": 2}; None = 1 người mỗi làn
)
p = t2b.Pipeline(BPMN_FILE, cfg=cfg)
print("Quy trình:", p.M["name"], "| làn:", p.M["lanes"], "| đầu ra:", p.out)
print("Luật:", p.cfg.rules, "| tự động hoá (C3):", p.cfg.automate or "–", "| sửa luồng (C0):", p.cfg.repair_flows or "–")
display(pd.DataFrame(t2b.DEFAULT_SCENARIOS).T)

## Bước 1 – Kiểm tra cấu trúc và soundness

BPMN hợp lệ **cú pháp** chưa chắc hợp lệ **hành vi**. Mô hình được chuyển sang mạng Petri và kiểm tra bằng Woflan;
đồng thời dò các lỗi hay gặp khi mô tả quy trình bằng lời: rẽ nhánh hoặc hội tụ ngầm ở một công việc, nhánh XOR thiếu
hoặc trùng nhãn, nhãn giữ chỗ chưa sửa, công việc của hệ thống đặt trong làn của người.

In [ ]:
r1 = p.step1_structure()
display(r1["elements"]); display(r1["flows"])
print("Sound (Woflan):", r1["sound"])
display(r1["issues"] if len(r1["issues"]) else "Không phát hiện vấn đề cấu trúc.")
print("Một số vết mô hình cho phép:"); display(r1["traces"])
show(f"{p.out}/fig_bpmn_AS_IS.png")

## Bước 2 – Nhật ký sự kiện

Ưu tiên nhật ký do text2BPMN xuất. Nếu chưa có, bộ mô phỏng chạy token trên chính BPMN với bốn kịch bản,
thêm hai yếu tố mà text2BPMN chưa có: hồ sơ đến theo thời gian và số người mỗi làn, để có hàng đợi và thời gian chờ.
Sau đó kiểm tra chất lượng: thiếu cột, giá trị rỗng, thời gian ngược, dòng trùng, hồ sơ bắt đầu hoặc kết thúc sai.

In [ ]:
r2 = p.step2_event_log()
print("Nguồn:", r2["source"], "| kích thước:", r2["shape"])
display(p.events.head(6))
display(r2["quality"])

## Bước 3 – Thống kê mô tả

*Thời gian làm việc* là tổng thời gian xử lý các bước, đúng chỉ số text2BPMN hiển thị. *Thời gian lưu trú* tính từ lúc
nộp đến lúc xong, gồm cả thời gian chờ. Bảng tỷ trọng dùng để đối chiếu với thanh "bước tốn thời gian nhất" của text2BPMN.

In [ ]:
r3 = p.step3_describe()
print("Thời hạn (giả định):", r3["sla"], "phút")
display(r3["kpi"]); display(r3["work_share"]); display(r3["top_variants"])

## Bước 4 – Khám phá quy trình
Đánh giá trên chính nhật ký dùng để khám phá (in-sample), nên các chỉ số có xu hướng lạc quan.

In [ ]:
r4 = p.step4_discovery()
display(r4["discovery"])
show(r4["dfg_images"][1] if len(r4["dfg_images"]) > 1 else r4["dfg_images"][0])

## Bước 5 – Kiểm tra tuân thủ

*Log move* là bước có trong nhật ký nhưng mô hình không cho phép (gồm cả làm lại, vì BPMN không có vòng lặp);
*model move* là bước mô hình đòi hỏi nhưng bị bỏ qua. Bảng luật cho tỷ lệ hồ sơ **vi phạm** từng luật.

In [ ]:
r5 = p.step5_conformance()
display(r5["fitness"]); display(r5["deviations"]); display(r5["rule_violations"])

## Bước 6 – Điểm nghẽn

Xếp hạng theo **tổng thời gian chờ** (bắt đầu − sẵn sàng), không theo thời gian làm việc. Mức tải từng làn giải thích
vì sao hàng đợi hình thành.

In [ ]:
r6 = p.step6_bottleneck()
for s, t in r6["tables"].items():
    print(f"=== {s} | tải làn: {r6['utilization'][s].to_dict()}"); display(t)
print(f"Điểm nghẽn ({r6['hot_scenario']}): {r6['top']} – làn {r6['top_lane']}" if r6["has_queue"]
      else "Không có hàng đợi: mọi bước là việc của hệ thống hoặc năng lực dư.")
show(f"{p.out}/fig_dfg_perf.png")   # biểu đồ matplotlib (heatmap, chờ/xử lý, hàng đợi) hiện ngay bên dưới

## Bước 7 – Yếu tố gắn với trễ hạn
Cây quyết định nông. Kết quả là liên kết thống kê, không phải quan hệ nhân quả; nhân quả được kiểm tra bằng can thiệp ở Bước 8.

In [ ]:
r7 = p.step7_root_cause()
print("Số hồ sơ:", r7["n"], "| tỷ lệ trễ hạn:", r7["ty_le_tre_han"], "| balanced accuracy:", r7["bal_acc"])
if r7["tree_text"]:
    print(r7["tree_text"]); display(r7["importance"])
display(r7["by_rework"]); display(r7["by_wip"])

## Bước 8 – BPMN TO-BE và đánh giá what-if

| Mã | Căn cứ | Thay đổi |
|---|---|---|
| C0 | Bước 1: BPMN không sound | Xoá các luồng thừa (chỉ khi cần) |
| C1 | Bước 6: làn nghẽn tải cao | Thêm một người cho làn nghẽn của từng kịch bản |
| C2 | Bước 5, 7: làm lại gắn với trễ hạn | Giả định giảm 50% làm lại ở bước làm lại nhiều nhất |
| C3 | Bước 6: bước thủ công tốn thời gian | Tự động hoá (đổi thành serviceTask trên BPMN) |

AS-IS và TO-BE dùng cùng dòng hồ sơ (common random numbers), lặp `n_rep` lần, báo trung bình ± khoảng tin cậy 95%
và kiểm định Wilcoxon theo cặp. Bước này chạy lâu nhất (khoảng 10–30 giây mỗi quy trình).

In [ ]:
r8 = p.step8_tobe()
print("Thay đổi:", r8["changes"], "| bước làm lại nhiều nhất:", r8["top_rework"], "| TO-BE sound:", r8["tobe_sound"])
display(r8["summary"]); display(r8["wilcoxon"])
print(f"Fitness log TO-BE trên BPMN TO-BE ({r8['verify_scenario']}):", r8["fitness_tobe"])
display(r8["wait_compare"])
show(f"{p.out}/fig_bpmn_TO_BE.png")

## Tải kết quả về
Nén thư mục đầu ra (bảng CSV, hình PNG, log CSV/XES, tệp BPMN TO-BE).

In [ ]:
archive = shutil.make_archive(p.out, "zip", p.out)
print("Đã tạo", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## Chạy cả ba quy trình cùng lúc (tuỳ chọn)

In [ ]:
RUN_ALL = False
if RUN_ALL:
    for f in sorted(glob.glob("data/bpmn/*.bpmn")):
        t2b.Pipeline(f, cfg=t2b.Config(n_rep=10)).run()